[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pyshka501/rl-textbook/blob/main/translations/ru/notebooks/ch12_dpo_alignment_ru.ipynb)

<div style="background: linear-gradient(135deg, #1B474D 0%, #944454 100%); padding: 30px; border-radius: 12px; margin-bottom: 20px;">
  <h1 style="color: white; margin: 0; font-size: 2em;">Глава 12. DPO и выравнивание</h1>
  <p style="color: #BCE2E7; margin-top: 10px; font-size: 1.1em;">
    Прямая оптимизация предпочтений (DPO) убирает отдельную модель вознаграждения и RL-цикл,
    напрямую оптимизируя стратегию по парам предпочтений людей.
    Выводим функцию потерь, реализуем её с нуля и исследуем роль $\beta$.
  </p>
</div>

**Цели обучения:**
- Вывести и реализовать функцию потерь DPO с нуля на PyTorch
- Обучить DPO на синтетических парах предпочтений с GPT-2 small
- Сравнить кривую потерь DPO с базовым PPO
- Визуализировать неявное вознаграждение $\beta \log(\pi / \pi_{\text{ref}})$
- Понять влияние $\beta$ на силу выравнивания

<div style="background: #1C1B19; border-left: 4px solid #20808D; padding: 15px; border-radius: 6px; margin: 10px 0;">
  <h2 style="color: #BCE2E7; margin: 0;">Подготовка окружения</h2>
  <p style="color: #CDCCCA; margin-top: 8px;">Рекомендуется T4 GPU. Ожидаемое суммарное время: ~10 минут.</p>
</div>

In [ ]:
!pip install -q transformers accelerate datasets trl torch matplotlib numpy tqdm

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import GPT2LMHeadModel, GPT2Tokenizer
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW
from tqdm.auto import tqdm
import copy, warnings, time
warnings.filterwarnings('ignore')

np.random.seed(42)
torch.manual_seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

plt.rcParams.update({
    'figure.facecolor': '#F7F6F2', 'axes.facecolor': '#F9F8F5',
    'axes.edgecolor': '#D4D1CA', 'axes.labelcolor': '#28251D',
    'text.color': '#28251D', 'xtick.color': '#7A7974',
    'ytick.color': '#7A7974', 'grid.color': '#D4D1CA',
    'font.family': 'sans-serif',
})

<div style="background: #1C1B19; border-left: 4px solid #A84B2F; padding: 15px; border-radius: 6px; margin: 15px 0;">
  <h2 style="color: #BCE2E7; margin: 0;">§12.1 Функция потерь DPO — вывод</h2>
</div>

### От PPO к DPO

Стандартный RLHF (на основе PPO) решает:

$$ \max_{\pi} \; \mathbb{E}_{y \sim \pi(\cdot|x)}[r_\phi(x,y)] - \beta \, D_{\text{KL}}[\pi \| \pi_{\text{ref}}] $$

DPO (Rafailov и др., 2023) замечает, что оптимальная стратегия в этой задаче равна:

$$ \pi^*(y|x) = \frac{\pi_{\text{ref}}(y|x)}{Z(x)} \exp\!\left(\frac{r(x,y)}{\beta}\right) $$

Обращая, вознаграждение можно выразить как:

$$ r(x,y) = \beta \log \frac{\pi^*(y|x)}{\pi_{\text{ref}}(y|x)} + \beta \log Z(x) $$

Подставляя в цель Брэдли\,---\,Терри и сокращая $Z(x)$, получаем **функцию потерь DPO**:

$$ \mathcal{L}_{\text{DPO}}(\pi_\theta; \pi_{\text{ref}}) = -\mathbb{E}_{(y_w,y_l)}\left[\log \sigma\!\left(\beta \log \frac{\pi_\theta(y_w|x)}{\pi_{\text{ref}}(y_w|x)} - \beta \log \frac{\pi_\theta(y_l|x)}{\pi_{\text{ref}}(y_l|x)}\right)\right] $$

Никакой отдельной модели вознаграждения. Никакого RL-цикла. Просто supervised-функция потерь по парам.

In [ ]:
# ── DPO Loss Implementation ──────────────────────────────────────────────

def log_prob_of_sequence(model, input_ids, attention_mask, labels):
    """Compute sum of log-probs for label tokens (ignoring -100 positions)."""
    with torch.no_grad() if not model.training else torch.enable_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    logits = outputs.logits  # (B, T, V)
    # Shift: predict position t+1 from position t
    shift_logits = logits[:, :-1, :]
    shift_labels = labels[:, 1:]
    log_probs = F.log_softmax(shift_logits, dim=-1)
    token_log_probs = log_probs.gather(2, shift_labels.unsqueeze(-1).clamp(min=0)).squeeze(-1)
    mask = (shift_labels != -100).float()
    return (token_log_probs * mask).sum(dim=-1)  # (B,)


def dpo_loss(policy_model, ref_model, batch, beta=0.1):
    """
    DPO loss.
    batch contains: chosen_ids, chosen_mask, chosen_labels,
                    rejected_ids, rejected_mask, rejected_labels
    """
    # Log-probs under policy
    lp_chosen_pol  = log_prob_of_sequence(policy_model,
        batch['chosen_ids'], batch['chosen_mask'], batch['chosen_labels'])
    lp_rejected_pol = log_prob_of_sequence(policy_model,
        batch['rejected_ids'], batch['rejected_mask'], batch['rejected_labels'])

    # Log-probs under frozen reference
    with torch.no_grad():
        lp_chosen_ref   = log_prob_of_sequence(ref_model,
            batch['chosen_ids'], batch['chosen_mask'], batch['chosen_labels'])
        lp_rejected_ref = log_prob_of_sequence(ref_model,
            batch['rejected_ids'], batch['rejected_mask'], batch['rejected_labels'])

    # Implicit reward margins
    reward_chosen   = beta * (lp_chosen_pol   - lp_chosen_ref)
    reward_rejected = beta * (lp_rejected_pol - lp_rejected_ref)

    # Bradley-Terry loss
    loss = -F.logsigmoid(reward_chosen - reward_rejected).mean()
    return loss, reward_chosen.detach(), reward_rejected.detach()


print('DPO loss function defined.')
print('Key insight: DPO reparameterises the reward as β·log(π_θ/π_ref)')
print('No reward model training required — the preference loss is applied directly to the LM.')

<div style="background: #1C1B19; border-left: 4px solid #A84B2F; padding: 15px; border-radius: 6px; margin: 15px 0;">
  <h2 style="color: #BCE2E7; margin: 0;">§12.2 Синтетический датасет предпочтений и обучение DPO</h2>
</div>

Строим синтетические пары предпочтений: для каждого промпта **выбранный** ответ информативнее/позитивнее, **отвергнутый** — короче или негативный. Затем обучаем GPT-2 с DPO несколько шагов, чтобы наблюдать динамику функции потерь.

In [ ]:
# ── Synthetic preference dataset ─────────────────────────────────────────
PREFERENCE_DATA = [
    {
        'prompt': 'Explain what reinforcement learning is.',
        'chosen': 'Reinforcement learning is a type of machine learning where an agent learns by interacting with an environment and receiving reward signals.',
        'rejected': 'I do not know.',
    },
    {
        'prompt': 'What is a neural network?',
        'chosen': 'A neural network is a computational model inspired by the brain, consisting of layers of interconnected nodes that learn to transform inputs into outputs.',
        'rejected': 'It is a thing.',
    },
    {
        'prompt': 'Describe the attention mechanism.',
        'chosen': 'The attention mechanism allows a model to focus on relevant parts of the input sequence when producing each output, computing a weighted sum of value vectors.',
        'rejected': 'Attention is a technique.',
    },
    {
        'prompt': 'What is the difference between supervised and unsupervised learning?',
        'chosen': 'Supervised learning uses labelled data to learn a mapping from inputs to outputs. Unsupervised learning finds patterns in unlabelled data without explicit labels.',
        'rejected': 'They are different types of learning.',
    },
    {
        'prompt': 'How does gradient descent work?',
        'chosen': 'Gradient descent iteratively updates model parameters by moving in the direction of the negative gradient of the loss, reducing prediction error step by step.',
        'rejected': 'It updates weights somehow.',
    },
    {
        'prompt': 'What is overfitting?',
        'chosen': 'Overfitting occurs when a model learns the training data too well, including noise, and fails to generalise to unseen examples, resulting in high test error.',
        'rejected': 'Bad training.',
    },
    {
        'prompt': 'Describe the transformer architecture.',
        'chosen': 'The transformer uses self-attention layers to process sequences in parallel, with positional encodings to capture order, enabling efficient long-range dependency modelling.',
        'rejected': 'A transformer is a model.',
    },
    {
        'prompt': 'What is a reward function in RL?',
        'chosen': 'A reward function maps state-action pairs to scalar signals, guiding the agent toward desirable behaviour by providing feedback on the quality of its actions.',
        'rejected': 'It gives scores.',
    },
]

tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token

def encode_pair(prompt, response, max_length=80):
    text = f"{prompt} {response}{tokenizer.eos_token}"
    enc = tokenizer(text, truncation=True, max_length=max_length,
                    padding='max_length', return_tensors='pt')
    labels = enc.input_ids.clone()
    # Mask prompt
    prompt_len = tokenizer(prompt, return_tensors='pt').input_ids.shape[1]
    labels[0, :prompt_len] = -100
    labels[enc.attention_mask == 0] = -100
    return enc.input_ids.squeeze(), enc.attention_mask.squeeze(), labels.squeeze()


class DPODataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data[idx]
        c_ids, c_mask, c_labels = encode_pair(row['prompt'], row['chosen'])
        r_ids, r_mask, r_labels = encode_pair(row['prompt'], row['rejected'])
        return {
            'chosen_ids': c_ids, 'chosen_mask': c_mask, 'chosen_labels': c_labels,
            'rejected_ids': r_ids, 'rejected_mask': r_mask, 'rejected_labels': r_labels,
        }


dpo_dataset = DPODataset(PREFERENCE_DATA * 8, tokenizer)  # repeat for more steps
dpo_loader  = DataLoader(dpo_dataset, batch_size=4, shuffle=True)
print(f'DPO dataset: {len(dpo_dataset)} pairs')

In [ ]:
# ── DPO Training (~2 min on T4) ───────────────────────────────────────────
BETA = 0.1
N_EPOCHS_DPO = 4

policy_model = GPT2LMHeadModel.from_pretrained('gpt2').to(DEVICE)
ref_model    = GPT2LMHeadModel.from_pretrained('gpt2').to(DEVICE)
ref_model.eval()  # frozen reference
for p in ref_model.parameters():
    p.requires_grad = False

optimizer = AdamW(policy_model.parameters(), lr=1e-5)

dpo_losses = []
chosen_rewards_log = []
rejected_rewards_log = []

for epoch in range(N_EPOCHS_DPO):
    policy_model.train()
    ep_loss = 0.0
    for batch in dpo_loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        loss, r_chosen, r_rejected = dpo_loss(policy_model, ref_model, batch, beta=BETA)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(policy_model.parameters(), 1.0)
        optimizer.step()
        optimizer.zero_grad()

        dpo_losses.append(loss.item())
        chosen_rewards_log.append(r_chosen.mean().item())
        rejected_rewards_log.append(r_rejected.mean().item())
        ep_loss += loss.item()

    print(f'Epoch {epoch+1}: avg DPO loss = {ep_loss/len(dpo_loader):.4f}')

print('DPO training complete.')

In [ ]:
# ── Mock PPO baseline for comparison ─────────────────────────────────────
# Simulate PPO loss curve: typically shows more variance and slower convergence
def mock_ppo_curve(n_steps, noise=0.12, decay=0.01, baseline=0.9):
    t = np.arange(n_steps)
    trend = baseline * np.exp(-decay * t)
    noise_arr = noise * np.random.randn(n_steps)
    return trend + noise_arr

n_steps = len(dpo_losses)
np.random.seed(5)
ppo_curve = mock_ppo_curve(n_steps, noise=0.15, decay=0.008, baseline=0.95)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Left: loss curves
smooth = lambda x, w=5: np.convolve(x, np.ones(w)/w, mode='valid')
steps = np.arange(n_steps)
axes[0].plot(steps, dpo_losses, color='#BCE2E7', alpha=0.4, linewidth=1)
axes[0].plot(range(4, n_steps), smooth(dpo_losses), color='#20808D', linewidth=2, label='DPO')
axes[0].plot(steps, ppo_curve, color='#FFC553', alpha=0.4, linewidth=1)
axes[0].plot(range(4, n_steps), smooth(ppo_curve), color='#A84B2F', linewidth=2,
             linestyle='--', label='PPO (mock)')
axes[0].set_xlabel('Training Step')
axes[0].set_ylabel('Loss')
axes[0].set_title('§12.2 — DPO vs PPO Loss Curves', fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.4)

# Right: implicit reward margin over training
margin = np.array(chosen_rewards_log) - np.array(rejected_rewards_log)
axes[1].plot(range(4, n_steps), smooth(margin), color='#20808D', linewidth=2)
axes[1].axhline(0, color='#7A7974', linestyle='--', linewidth=1)
axes[1].fill_between(range(4, n_steps), 0, smooth(margin),
                     alpha=0.2, color='#20808D', label='Chosen − Rejected reward')
axes[1].set_xlabel('Training Step')
axes[1].set_ylabel('Implicit Reward Margin β·log(π/π_ref)')
axes[1].set_title('Implicit Reward: Chosen − Rejected', fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.4)

plt.tight_layout()
plt.savefig('ch12_fig1_dpo_training.png', dpi=120, bbox_inches='tight')
plt.show()

<div style="background: #1C1B19; border-left: 4px solid #A84B2F; padding: 15px; border-radius: 6px; margin: 15px 0;">
  <h2 style="color: #BCE2E7; margin: 0;">§12.3 Неявное вознаграждение и влияние $\beta$</h2>
</div>

DPO определяет **неявное вознаграждение**:

$$ \hat{r}(x, y) = \beta \log \frac{\pi_\theta(y|x)}{\pi_{\text{ref}}(y|x)} $$

Гиперпараметр $\beta$ управляет компромиссом:
- **Малое $\beta$**: стратегия может далеко уйти от референса — сильнее выравнивание, но меньше разнообразия.
- **Большое $\beta$**: стратегия близка к референсу — больше разнообразия, но слабее выравнивание.

Теоретически $\beta$ — это коэффициент KL-регуляризации в RLHF-цели.

In [ ]:
# Compute implicit reward for one sample under different beta values
policy_model.eval()

sample = PREFERENCE_DATA[0]
c_ids, c_mask, c_labels = encode_pair(sample['prompt'], sample['chosen'])
r_ids, r_mask, r_labels = encode_pair(sample['prompt'], sample['rejected'])

c_ids = c_ids.unsqueeze(0).to(DEVICE)
c_mask = c_mask.unsqueeze(0).to(DEVICE)
c_labels = c_labels.unsqueeze(0).to(DEVICE)
r_ids = r_ids.unsqueeze(0).to(DEVICE)
r_mask = r_mask.unsqueeze(0).to(DEVICE)
r_labels = r_labels.unsqueeze(0).to(DEVICE)

with torch.no_grad():
    lp_c_pol = log_prob_of_sequence(policy_model, c_ids, c_mask, c_labels).item()
    lp_r_pol = log_prob_of_sequence(policy_model, r_ids, r_mask, r_labels).item()
    lp_c_ref = log_prob_of_sequence(ref_model, c_ids, c_mask, c_labels).item()
    lp_r_ref = log_prob_of_sequence(ref_model, r_ids, r_mask, r_labels).item()

log_ratio_chosen   = lp_c_pol - lp_c_ref
log_ratio_rejected = lp_r_pol - lp_r_ref

betas = np.linspace(0.01, 1.0, 100)
reward_chosen_beta   = betas * log_ratio_chosen
reward_rejected_beta = betas * log_ratio_rejected
margin_beta = reward_chosen_beta - reward_rejected_beta

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Left: rewards vs beta
axes[0].plot(betas, reward_chosen_beta, color='#20808D', linewidth=2.5, label='Chosen response')
axes[0].plot(betas, reward_rejected_beta, color='#A84B2F', linewidth=2.5,
             linestyle='--', label='Rejected response')
axes[0].axhline(0, color='#7A7974', linewidth=0.8, linestyle=':')
axes[0].set_xlabel('β (KL regularisation strength)')
axes[0].set_ylabel('Implicit Reward β·log(π/π_ref)')
axes[0].set_title('§12.3 — Implicit Reward vs β', fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.4)

# Right: reward margin (chosen - rejected) vs beta
axes[1].plot(betas, margin_beta, color='#1B474D', linewidth=2.5)
axes[1].fill_between(betas, 0, margin_beta,
                     where=(margin_beta > 0), alpha=0.2, color='#20808D', label='Alignment gap > 0')
axes[1].fill_between(betas, margin_beta, 0,
                     where=(margin_beta < 0), alpha=0.2, color='#A84B2F', label='Alignment gap < 0')
axes[1].axhline(0, color='#7A7974', linewidth=1, linestyle='--')
axes[1].set_xlabel('β')
axes[1].set_ylabel('Reward Margin (Chosen − Rejected)')
axes[1].set_title('Alignment Strength vs β', fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.4)

plt.tight_layout()
plt.savefig('ch12_fig2_beta_effect.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'log(π_policy / π_ref) for chosen  : {log_ratio_chosen:.4f}')
print(f'log(π_policy / π_ref) for rejected: {log_ratio_rejected:.4f}')
print(f'Margin at β=0.1: {0.1 * (log_ratio_chosen - log_ratio_rejected):.4f}')

In [ ]:
# ── Visualise reward distribution across all pairs ────────────────────────
chosen_rewards_final   = []
rejected_rewards_final = []

policy_model.eval()
with torch.no_grad():
    for row in PREFERENCE_DATA:
        c_ids_b, c_mask_b, c_lbl_b = encode_pair(row['prompt'], row['chosen'])
        r_ids_b, r_mask_b, r_lbl_b = encode_pair(row['prompt'], row['rejected'])
        for ids, mask, lbl, store in [
            (c_ids_b, c_mask_b, c_lbl_b, chosen_rewards_final),
            (r_ids_b, r_mask_b, r_lbl_b, rejected_rewards_final)
        ]:
            pol = log_prob_of_sequence(policy_model,
                ids.unsqueeze(0).to(DEVICE), mask.unsqueeze(0).to(DEVICE), lbl.unsqueeze(0).to(DEVICE)).item()
            ref = log_prob_of_sequence(ref_model,
                ids.unsqueeze(0).to(DEVICE), mask.unsqueeze(0).to(DEVICE), lbl.unsqueeze(0).to(DEVICE)).item()
            store.append(BETA * (pol - ref))

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(chosen_rewards_final, bins=8, alpha=0.75, color='#20808D', label='Chosen', edgecolor='white')
ax.hist(rejected_rewards_final, bins=8, alpha=0.75, color='#A84B2F', label='Rejected', edgecolor='white')
ax.axvline(np.mean(chosen_rewards_final), color='#1B474D', linewidth=2, linestyle='--',
           label=f'Mean chosen: {np.mean(chosen_rewards_final):.3f}')
ax.axvline(np.mean(rejected_rewards_final), color='#944454', linewidth=2, linestyle='--',
           label=f'Mean rejected: {np.mean(rejected_rewards_final):.3f}')
ax.set_xlabel('Implicit Reward β·log(π_θ/π_ref)')
ax.set_ylabel('Count')
ax.set_title('§12.3 — Implicit Reward Distribution After DPO', fontweight='bold')
ax.legend(fontsize=9)
ax.grid(alpha=0.4)
plt.tight_layout()
plt.savefig('ch12_fig3_reward_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

<div style="background: #1C1B19; border-left: 4px solid #20808D; padding: 20px; border-radius: 6px; margin: 15px 0;">
  <h2 style="color: #BCE2E7; margin: 0 0 12px 0;">Сводка и ключевые выводы</h2>
  <ul style="color: #D4D1CA; line-height: 1.8; margin: 0; padding-left: 22px;">
    <li><strong>Функция потерь DPO</strong> напрямую оптимизирует пары предпочтений как $-\log\sigma\left(\beta\left[\log\frac{\pi_\theta(y_w)}{\pi_{\text{ref}}(y_w)} - \log\frac{\pi_\theta(y_l)}{\pi_{\text{ref}}(y_l)}\right]\right)$, так что выравнивание можно обучать без явного RL-цикла.</li>
    <li><strong>Неявное вознаграждение</strong> возникает из логарифма отношения стратегии к референсу, $\hat{r}(x,y) = \beta\log(\pi_\theta / \pi_{\text{ref}})$, и после обучения присваивает большие значения предпочтительным ответам.</li>
    <li><strong>$\beta$</strong> управляет компромиссом выравнивание-референс: меньшее $\beta$ сильнее тянет к предпочтительным ответам и дальше от референсной модели; большее $\beta$ удерживает стратегию ближе к референсу.</li>
    <li><strong>DPO vs PPO</strong> — компромисс между простотой и гибкостью: DPO обычно сходится быстрее с меньшей дисперсией; PPO позволяет явный шейпинг вознаграждения и более широкий RL-контроль.</li>
  </ul>
</div>

<div style="background: #1e3a5f; padding: 15px; border-radius: 8px; border-left: 4px solid #f0a500; margin-top: 20px;">
  <strong style="color: #f0a500;">Дальше:</strong> в главе 13 — GRPO: групповая относительная оптимизация стратегии для задач рассуждения.
</div>